# Topic 8: The Barclays Complaint Agent (Capstone)

Barclays Customer Support Intelligence System | Topic 8 | CAPSTONE

This is the capstone. Across Topics 1 to 7 you built every piece of a
complaint-intelligence system: you loaded pretrained models, fine-tuned one,
transfer-learned another, adapted one with LoRA, and compressed a model for
serving. Each topic saved its work to S3.

Today you connect them. You will build ONE agent - a pure-Python program, no
agent frameworks - that uses an LLM as its brain and calls your earlier models
as tools to triage a Barclays complaint end to end.

## What this capstone is

This notebook hands you the setup: installs, environment, the S3 load of your
prior work, and the bare agent loop. Everything else - the tools, the tool
schemas, the system prompt, the orchestration - you write yourself, from
scratch. There are no numbered steps and no fill-in-the-blank scaffolds. You
have spent seven topics earning this.

## What you will build

A "Barclays Complaint Agent" that, given a raw customer complaint, decides on
its own which of its tools to call, calls them, and returns a triage decision
(category, urgency, a recommended action) with its reasoning.

## Estimated time

90 minutes. This is the hardest and most open task of the course.


## Section 0 - Environment Setup

The next four cells are GIVEN. Run them in order, top to bottom, and do not
edit them. They install dependencies, disable the TensorFlow backend, import
what you need, and load the artifacts your earlier topics saved to S3.


In [ ]:
# Disable the TensorFlow backend in transformers (SageMaker image compatibility).
# Must run before any transformers import.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [ ]:
# Capstone install cell. Pinned to the course-wide matrix.
!pip install -q \
    "openai>=1.0.0" \
    "transformers>=4.53,<4.54" \
    "accelerate>=1.0.0" \
    "tokenizers>=0.21,<0.22" \
    "sagemaker>=2.200.0,<3.0.0" \
    "numpy<2"

print("RESTART KERNEL before continuing -- environment packages were installed/upgraded.")


In [ ]:
# Imports, the OpenAI client, and the course S3 bucket. GIVEN - do not edit.
import json
import getpass

from openai import OpenAI
import sagemaker

# gpt-4o is the course brain model. The key is entered at runtime, never hardcoded.
_api_key = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=_api_key)
BRAIN_MODEL = "gpt-4o"

# The course S3 bucket - every topic saved its handoff artifacts here.
bucket = sagemaker.Session().default_bucket()
COURSE_PREFIX = "barclays-course"

print("OpenAI client ready. Brain model:", BRAIN_MODEL)
print("Course bucket:", bucket)


## Section 1 - What Your Earlier Topics Left You

Every required topic wrote a handoff artifact to
`s3://<bucket>/barclays-course/topic_<N>/`. The cell below loads them. This is
the raw material your agent's tools will stand on:

- Topic 1 - the triage system prompt and the test complaints.
- Topic 3 - the routing label set and a labelled complaint dataset.
- Topic 4 - a fully fine-tuned classifier (model pointer).
- Topic 5 - a transfer-learned classifier (model pointer).
- Topic 6 - a PEFT / LoRA-adapted classifier (model pointer).
- Topic 7 - the compressed, deployment-ready model and endpoint.

The load cell has a fallback: if you skipped a topic or a training job did not
finish, the artifact may be missing. That is expected and handled. Your tools
must cope with a missing artifact - the design guidance below tells you how.


In [ ]:
# Load every prior topic's handoff artifact. GIVEN - do not edit.
# Anything missing comes back as None; your tools must handle that gracefully.
import boto3, botocore

def handoff_read(topic_n, artifact):
    key = f"{COURSE_PREFIX}/topic_{topic_n}/{artifact}"
    try:
        body = boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()
        print(f"Loaded s3://{bucket}/{key}")
        return json.loads(body)
    except botocore.exceptions.ClientError:
        print(f"MISSING s3://{bucket}/{key} (a tool fallback will cover this)")
        return None

prior_work = {
    "triage":     handoff_read(1, "triage_config.json"),
    "labels":     handoff_read(3, "routing_labels.json"),
    "dataset":    handoff_read(3, "labelled_dataset.json"),
    "finetuned":  handoff_read(4, "model_pointer.json"),
    "transfer":   handoff_read(5, "model_pointer.json"),
    "peft":       handoff_read(6, "model_pointer.json"),
    "deployment": handoff_read(7, "deployment.json"),
}

# A sensible default routing label set if Topic 3 was skipped.
ROUTING_LABELS = (prior_work["labels"] or {}).get("labels") or [
    "fraud and security", "billing and charges",
    "account access", "general enquiry",
]
print()
print("Routing labels:", ROUTING_LABELS)
print("Prior-work artifacts present:",
      [k for k, v in prior_work.items() if v is not None])


## Section 2 - The Agent Loop

An agent is not magic. It is a loop:

1. Send the conversation and the list of available tools to the LLM.
2. The LLM replies. Either it answers, or it asks to call one or more tools.
3. If it asked for tools: run them yourself, append each result to the
   conversation, and go back to step 1.
4. If it answered: stop and return the answer.

The LLM never runs the tools. It only decides which to call and with what
arguments. Your code does the running.

<!-- DIAGRAM: the agent loop - LLM decides, code runs tools, results return, repeat until the LLM answers -->
[View diagram: the agent loop](../../plans/topic_8/diagrams/agent-loop.mmd)
> Diagram coming via /build-diagrams - the file above will contain the Mermaid source once built.

The diagram shows the cycle: LLM call, then a branch on whether the LLM asked
for tool calls; if it did, the code runs the tools, appends the results, and
loops back; if it did not, the loop exits with the final answer.

The next cell GIVES you this loop. It is plumbing, not the point of the
capstone - so you do not have to write it. Read it carefully: it expects two
things YOU will define later - a list called `TOOL_SCHEMAS` (the JSON
descriptions the LLM sees) and a dict called `TOOL_FUNCTIONS` (mapping each
tool name to the Python function that runs it).


In [ ]:
# The agent loop. GIVEN - do not edit. It drives the conversation and the
# tool calls. It depends on TOOL_SCHEMAS and TOOL_FUNCTIONS, which YOU define.

def run_agent(system_prompt, user_message, max_turns=8):
    """Run the agent loop. Returns (final_answer, full_message_history).

    full_message_history is the agent's memory: every message, tool call, and
    tool result, in order. The loop re-sends it every turn.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=BRAIN_MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        choice = response.choices[0].message
        messages.append(choice)

        if not choice.tool_calls:
            return choice.content, messages

        for call in choice.tool_calls:
            fn = TOOL_FUNCTIONS.get(call.function.name)
            args = json.loads(call.function.arguments)
            if fn is None:
                result = f"ERROR: unknown tool {call.function.name}"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"ERROR running {call.function.name}: {e}"
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })
    return "Agent stopped: reached the turn limit.", messages


## Section 3 - Your Capstone

Everything above was setup. From here on, you write the code.

### The goal

Build the Barclays Complaint Agent. Given a raw customer complaint string, it
must decide for itself which tools to use, call them, and return a triage
result: a routing category, an urgency judgement, and a recommended action,
with the reasoning that led there.

### What you must produce

Three TOOLS, their SCHEMAS, a SYSTEM PROMPT, and the wiring - then a run.

1. Tool A - a code / computation tool. The agent can hand it a small piece of
   Python (or a fixed analysis) to compute something about complaint data -
   for example, how many complaints in the loaded Topic 3 dataset fall in each
   category. You decide its exact contract.

2. Tool B - a classifier built from a model you trained earlier. Wrap the
   Topic 4 fine-tuned model OR the Topic 5 transfer-learned model. Load the
   real artifact from its S3 model pointer if it is present in `prior_work`.
   If it is missing, fall back to a small public HuggingFace classifier doing
   the same job (a `transformers` pipeline, `framework="pt"`). Either way the
   tool returns a routing category for a complaint.

3. Tool C - a second classifier, from your PEFT / LoRA work (Topic 6). Same
   load-or-fallback pattern. The agent now has two classifiers and must decide
   which to trust, or call both and compare - that decision is part of the
   capstone.

4. TOOL_SCHEMAS - a list of JSON tool descriptions, one per tool, in the
   OpenAI function-calling format. This is what the LLM sees.

5. TOOL_FUNCTIONS - a dict mapping each tool name to its Python function.

6. A SYSTEM PROMPT that makes gpt-4o behave as the Barclays triage agent:
   tells it its job, its tools, and what a finished answer looks like.

7. Call `run_agent(system_prompt, complaint)` on at least two of the Topic 1
   test complaints and show the agent's reasoning and final triage.

### Acceptance criteria

- Run the agent on a fraud-type complaint and a billing-type complaint.
- For each, the agent must call at least one classifier tool and return a
  routing category from `ROUTING_LABELS`, an urgency, and a recommended action.
- The full message history (the agent's memory) must show the tool calls and
  their results - the decision must be traceable, not a black box.
- A missing S3 artifact must not crash the agent: the fallback path must work.

### Constraints

- Pure Python and the `openai` / `transformers` SDKs only. No agent frameworks
  (no LangChain, no CrewAI, no LlamaIndex).
- The brain is `gpt-4o` via the `client` already created. Do not hardcode keys.
- Use the GIVEN `run_agent` loop as-is. Your job is the tools and the brains
  around it, not the loop.
- HuggingFace pipelines must pass `framework="pt"` (SageMaker TF-backend rule).

### How you are assessed

There is no verification cell that grades you. The capstone works when you can
run it end to end and read a sensible, traceable triage decision out of the
agent. That is the bar. You have no scaffolding and no hints - this is the
point of a capstone.


### Before you write code - plan (5 minutes)

With the person next to you, agree on:

- Each tool's exact input and output. A tool that returns a clean string the
  LLM can read is far easier to orchestrate than one that returns a raw object.
- What goes in the system prompt so the agent actually USES the tools instead
  of guessing. What happens if you do not mention a tool in the prompt?
- The order you expect a good agent to call tools in - and whether you should
  force that order or let the LLM decide. Which is more in the spirit of an
  agent?
- How you will SEE the agent's reasoning. The returned message history is your
  window into it - plan how you will print it readably.


## Reference Implementation

Below is one complete, working reference implementation of the capstone. It is
broken into a few cells with a short note before each. Yours may differ - any
solution that meets the acceptance criteria is correct. This one keeps every
tool returning a plain string, because a string is the easiest thing for the
LLM to read back.


### Tool A - the computation tool

`analyze_complaint_dataset` counts complaints per routing category in the
Topic 3 labelled dataset. If that dataset was not loaded it returns a plain
message instead of crashing - the agent can read that and move on.


In [ ]:
# Tool A - a computation tool over the Topic 3 labelled dataset.

def analyze_complaint_dataset(category=None):
    """Count complaints per routing category in the Topic 3 dataset.

    If category is given, return just that category's count. Returns a plain
    string so the LLM can read it directly.
    """
    dataset = prior_work.get("dataset")
    if not dataset:
        return ("The Topic 3 labelled dataset is not available, so no dataset "
                "statistics can be computed. Proceed using the classifier tools.")

    # The dataset is a list of {"text": ..., "label": ...} records.
    records = dataset.get("records") if isinstance(dataset, dict) else dataset
    counts = {}
    for row in records:
        label = row.get("label", "unknown")
        counts[label] = counts.get(label, 0) + 1

    if category is not None:
        return f"Category '{category}' has {counts.get(category, 0)} complaints in the dataset."

    lines = [f"  {label}: {n}" for label, n in sorted(counts.items())]
    return "Complaint counts per category:\n" + "\n".join(lines)


### The shared classifier loader

Both classifier tools need the same load-or-fallback logic, so it lives in one
helper. If the S3 model pointer is present and names a real model id we use it;
otherwise we load a small public HuggingFace model. Every pipeline is built
with `framework="pt"` because the SageMaker image must not touch TensorFlow.


In [ ]:
# Shared helper: load a text-classification pipeline from a model pointer,
# or fall back to a public model when the pointer is missing.
from transformers import pipeline

# Cache pipelines so the agent does not reload a model on every tool call.
_pipeline_cache = {}

def _load_classifier(pointer, fallback_model_id):
    """Return a transformers text-classification pipeline.

    pointer is a dict from prior_work (or None). If it carries a usable model
    reference we use it; otherwise we load fallback_model_id. framework='pt'
    is mandatory for the SageMaker image.
    """
    # Decide which model id to load.
    model_id = fallback_model_id
    note = f"fallback model '{fallback_model_id}'"
    if pointer:
        # A real Topic 4/5/6 pointer may carry a hub id or a tar uri.
        candidate = pointer.get("model_id") or pointer.get("hf_model_id")
        tar_uri = pointer.get("model_tar_uri")
        if candidate:
            model_id = candidate
            note = f"trained model '{candidate}' from the S3 pointer"
        elif tar_uri:
            # For the capstone we note the artifact and still load a public
            # model - unpacking a SageMaker tar is out of scope here.
            note = (f"S3 artifact {tar_uri} noted; loading fallback "
                    f"'{fallback_model_id}' for the capstone run")

    if model_id not in _pipeline_cache:
        _pipeline_cache[model_id] = pipeline(
            "text-classification",
            model=model_id,
            framework="pt",
        )
    return _pipeline_cache[model_id], note


### Tool B - the fine-tuned / transfer-learned classifier

Tool B prefers the Topic 4 fine-tuned pointer and falls back to the Topic 5
transfer-learned pointer, then to a public model. It maps the raw pipeline
label onto one of `ROUTING_LABELS` so the agent always gets a category it
recognises.


In [ ]:
# Tool B - classifier from the Topic 4 fine-tuned (or Topic 5 transfer) model.

def _map_to_routing_label(raw_label):
    """Map an arbitrary pipeline label onto one of ROUTING_LABELS."""
    text = str(raw_label).lower()
    for label in ROUTING_LABELS:
        # A loose match: any routing-label word appearing in the raw label.
        if any(word in text for word in label.split()):
            return label
    # Sentiment-style fallback: negative complaints lean to fraud/security.
    if "neg" in text or "1" in text:
        return ROUTING_LABELS[0]
    return ROUTING_LABELS[-1]

def classify_finetuned(complaint):
    """Classify a complaint with the fine-tuned model. Returns a routing category."""
    pointer = prior_work.get("finetuned") or prior_work.get("transfer")
    clf, note = _load_classifier(pointer, "distilbert-base-uncased-finetuned-sst-2-english")
    prediction = clf(complaint[:512])[0]
    category = _map_to_routing_label(prediction["label"])
    return (f"Fine-tuned classifier ({note}) routed this complaint to "
            f"'{category}' (raw label {prediction['label']}, "
            f"score {prediction['score']:.2f}).")


### Tool C - the PEFT / LoRA classifier

Tool C is the same pattern with the Topic 6 PEFT pointer. Giving the agent two
classifiers is deliberate: the system prompt tells it to call both and compare,
and to flag any disagreement in its reasoning.


In [ ]:
# Tool C - classifier from the Topic 6 PEFT / LoRA-adapted model.

def classify_peft(complaint):
    """Classify a complaint with the PEFT/LoRA model. Returns a routing category."""
    pointer = prior_work.get("peft")
    clf, note = _load_classifier(pointer, "distilbert-base-uncased-finetuned-sst-2-english")
    prediction = clf(complaint[:512])[0]
    category = _map_to_routing_label(prediction["label"])
    return (f"PEFT/LoRA classifier ({note}) routed this complaint to "
            f"'{category}' (raw label {prediction['label']}, "
            f"score {prediction['score']:.2f}).")


### The tool schemas and the function map

`TOOL_SCHEMAS` is what the LLM sees - one JSON description per tool, in OpenAI
function-calling format. `TOOL_FUNCTIONS` maps each tool name to the Python
function the loop actually runs. The names must match exactly across both.


In [ ]:
# TOOL_SCHEMAS - the JSON descriptions the LLM sees.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "analyze_complaint_dataset",
            "description": ("Count complaints per routing category in the "
                            "historical Barclays complaint dataset. Use this "
                            "for context on how common a category is."),
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "description": "Optional. A single routing category to count.",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "classify_finetuned",
            "description": ("Classify a complaint into a routing category using "
                            "the fine-tuned classifier. Returns the category."),
            "parameters": {
                "type": "object",
                "properties": {
                    "complaint": {
                        "type": "string",
                        "description": "The raw complaint text to classify.",
                    },
                },
                "required": ["complaint"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "classify_peft",
            "description": ("Classify a complaint into a routing category using "
                            "the PEFT/LoRA classifier. Returns the category."),
            "parameters": {
                "type": "object",
                "properties": {
                    "complaint": {
                        "type": "string",
                        "description": "The raw complaint text to classify.",
                    },
                },
                "required": ["complaint"],
            },
        },
    },
]

# TOOL_FUNCTIONS - maps each tool name to the Python function that runs it.
TOOL_FUNCTIONS = {
    "analyze_complaint_dataset": analyze_complaint_dataset,
    "classify_finetuned": classify_finetuned,
    "classify_peft": classify_peft,
}

print("Tools registered:", list(TOOL_FUNCTIONS))


### The system prompt

The system prompt is what turns gpt-4o into the Barclays triage agent. It names
the three tools, tells the agent to use both classifiers and compare them, and
defines exactly what a finished answer must contain.


In [ ]:
# SYSTEM_PROMPT - turns gpt-4o into the Barclays triage agent.
SYSTEM_PROMPT = f"""You are the Barclays Complaint Triage Agent.

Your job: given a raw customer complaint, decide its routing category, judge
its urgency, and recommend an action.

You have three tools:
- analyze_complaint_dataset: counts historical complaints per category.
- classify_finetuned: a fine-tuned classifier that returns a routing category.
- classify_peft: a PEFT/LoRA classifier that returns a routing category.

How to work:
1. Call BOTH classify_finetuned and classify_peft on the complaint.
2. If they agree, trust that category. If they disagree, pick the more
   plausible one given the complaint text and say why in your reasoning.
3. Optionally call analyze_complaint_dataset for context on the category.
4. The routing category MUST be one of: {ROUTING_LABELS}.

A finished answer is a short report with exactly these four parts:
- Category: one of the routing labels above.
- Urgency: low, medium, or high.
- Recommended action: one concrete next step for the support team.
- Reasoning: two or three sentences, including which tools you called and
  whether the classifiers agreed.

Do not answer until you have called the classifier tools."""

print("System prompt ready.")


### Run the agent

Now run `run_agent` on two Topic 1 test complaints - one fraud-type, one
billing-type - and pretty-print the final triage plus the tool-call trace from
the message history. The trace is the proof that the decision is traceable.


In [ ]:
# Run the agent on two test complaints and show the reasoning trace.

# Use the Topic 1 test complaints if they were loaded; otherwise use built-in ones.
_default_complaints = [
    ("Someone used my Barclays debit card in another country and I never "
     "authorised it. There are three charges I do not recognise and I am scared."),
    ("I have been charged a 35 pound overdraft fee twice this month even "
     "though I was told the fee was waived. I want it refunded."),
]
triage_cfg = prior_work.get("triage") or {}
test_complaints = triage_cfg.get("test_complaints") or _default_complaints

def show_trace(history):
    """Print the tool calls and tool results from an agent message history."""
    for msg in history:
        # Assistant messages may carry tool calls (object form from the SDK).
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for call in tool_calls:
                print(f"  TOOL CALL  -> {call.function.name}({call.function.arguments})")
        # Tool result messages are plain dicts appended by the loop.
        if isinstance(msg, dict) and msg.get("role") == "tool":
            print(f"  TOOL RESULT-> {msg['content'][:160]}")

for i, complaint in enumerate(test_complaints[:2], 1):
    print("=" * 70)
    print(f"COMPLAINT {i}: {complaint[:120]}...")
    print("-" * 70)
    final_answer, history = run_agent(SYSTEM_PROMPT, complaint)
    print("AGENT TRACE:")
    show_trace(history)
    print("-" * 70)
    print("FINAL TRIAGE:")
    print(final_answer)
    print()


## Wrap-Up - You Built the Whole System

If your agent runs and returns a traceable triage decision, you have just
connected every topic of this course into one working program:

- The LLM brain is the prompt-engineering and LLM-API work from Topics 1-2.
- The classifier tools are the models you fine-tuned, transfer-learned, and
  LoRA-adapted in Topics 4-6.
- The graceful fallbacks are the HuggingFace ecosystem skills from Topic 3.
- The whole thing is deployable because of the compression work in Topic 7.

You did this with pure Python and one loop. No framework. You now know what an
agent actually is - a loop, some tools, and a model that decides - because you
built one with nothing hidden from you.

### Homework Extension - give the agent a fourth tool

Add a naive RAG tool: take a handful of internal Barclays policy snippets
(plain strings), embed them and the query, score with cosine similarity,
return the top-2 snippets as context the agent can cite. No vector database -
a list and a dot product. Wire it in as a fourth tool and re-run the agent on
a complaint whose answer depends on a policy.
